# NorthStar Navigator

### A Plain-Language Government Benefits Navigator for Minnesota

![NorthStar Navigator Banner](notebook_images/NorthStar_Navigator_banner.png)

**Kaggle Gemma 4 Good Hackathon** | [Live Demo](https://vcae8sqtjiyxrc-7860.proxy.runpod.net) | [GitHub](https://github.com/wanderduck/northstar_navigator) | [GGUF Model](https://huggingface.co/wanderduck/northstar-navigator-gguf)

---

NorthStar Navigator helps Minnesota residents understand which government assistance programs they may be eligible for. Users describe their situation in plain language, and the Navigator responds with personalized, actionable guidance at an appropriate reading level.

**Key Features:**
- Fine-tuned **Gemma 4 E4B** (4B params) via QLoRA + Unsloth, served through **Ollama**
- Three-stage pipeline: **Intake** (situation parsing) -> **Eligibility** (rule-based + RAG) -> **Response** (plain-language generation)
- **Multilingual**: English, Spanish, Hmong, and Somali
- **RAG knowledge base**: 1,938 documents from DHS Combined Manual, county programs, and SAM.gov federal listings
- Adjustable reading levels (simple / standard / detailed)
- Privacy-first: all inference runs locally via Ollama

**Prize Tracks:** Main Track, Digital Equity Impact, Safety & Trust Impact, Ollama Special Technology, Unsloth Special Technology

<div>
  <!-- Full-width top row -->
  <div style="padding: 10px; border-bottom: 1px solid #ccc;">
    <h2>Title Spanning Both Columns</h2>
    <p>This row sits on top and covers the full width of the layout.</p>
  </div>

  <!-- Two-column row -->
  <div style="display: flex;">
    <div style="flex: 1; padding: 10px;">
      <h3>Column 1</h3>
      <p>Text for the first column goes here.</p>
    </div>
    <div style="flex: 1; padding: 10px;">
      <h3>Column 2</h3>
      <p>Text for the second column goes here.</p>
    </div>
  </div>
</div>


## 1. Architecture

```
User: "I'm a single mom with two kids, just lost my job in Ramsey County..."
                            |
                    +-------v--------+
                    |   Stage 1:     |
                    |   INTAKE       |  Gemma 4 E4B extracts: household_size=3,
                    |   (LLM Parse)  |  income=$0, county=Ramsey, needs=[food, housing]
                    +-------+--------+
                            |
                    +-------v--------+
                    |   Stage 2:     |
                    |   ELIGIBILITY  |  Rule engine + RAG retrieval matches:
                    |   (Rules+RAG)  |  SNAP, MFIP, Emergency Assistance, WIC...
                    +-------+--------+
                            |
                    +-------v--------+
                    |   Stage 3:     |
                    |   RESPONSE     |  Gemma 4 E4B generates plain-language
                    |   (LLM Gen)    |  response at user's reading level
                    +-------+--------+
                            |
                            v
"Based on your situation, you may be eligible for several programs..."
```

**Technology Stack:**
- **Model**: Gemma 4 E4B, fine-tuned with QLoRA via Unsloth on A100-80GB
- **Quantization**: GGUF q4_k_m (5.3GB) for efficient inference
- **Serving**: Ollama (local inference, no API calls, full privacy)
- **RAG**: ChromaDB + sentence-transformers/all-MiniLM-L6-v2 + BM25 hybrid retrieval
- **UI**: Gradio with streaming chat interface
- **Deployment**: RunPod GPU Pod (RTX 4090)

## 2. Setup

Install dependencies and configure the Ollama model server. This notebook is designed to run on **Kaggle with a T4 GPU**.

In [ ]:
%%bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# Install Navigator dependencies
pip install -q gradio chromadb sentence-transformers rank-bm25 \
    textstat ollama pydantic httpx "huggingface_hub[cli]"

# Download the fine-tuned GGUF model from HuggingFace Hub
hf download wanderduck/northstar-navigator-gguf model-q4_k_m.gguf Modelfile --local-dir /tmp/gguf

echo "Setup complete"

In [ ]:
import subprocess, time, json, textwrap
import httpx

# Start Ollama server in background
env = {**__import__("os").environ, "OLLAMA_HOST": "0.0.0.0:11434"}
proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait for readiness
for i in range(30):
    try:
        r = httpx.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama ready after {i+1}s")
            break
    except Exception:
        pass
    time.sleep(1)

# Create the Navigator model from GGUF
result = subprocess.run(
    ["ollama", "create", "navigator", "-f", "/tmp/gguf/Modelfile"],
    capture_output=True, text=True, timeout=300,
    cwd="/tmp/gguf"
)
print(result.stdout or result.stderr)

# Pre-warm: load model into GPU memory
r = httpx.post("http://localhost:11434/api/generate",
    json={"model": "navigator", "prompt": "hello", "stream": False, "options": {"num_predict": 1}},
    timeout=120)
print(f"Model loaded into VRAM. Status: {r.status_code}")

## 3. Direct Model Interaction

Let's first interact with the fine-tuned Gemma 4 E4B model directly through Ollama to see how it responds to benefits-related queries.

In [ ]:
import ollama

def ask_navigator(prompt, temperature=1.0):
    """Send a prompt to the Navigator model and print the response."""
    response = ollama.chat(
        model="navigator",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": temperature, "top_p": 0.95, "num_predict": 512},
    )
    text = response["message"]["content"]
    print(textwrap.fill(text, width=90))
    return text

# English query
print("=" * 90)
print("QUERY: Single mother, laid off, Ramsey County")
print("=" * 90)
response_en = ask_navigator(
    "I'm a single mom with two kids, ages 3 and 7. I just got laid off "
    "from my warehouse job where I made $32,000. We're in Ramsey County "
    "and I'm worried about paying rent and feeding my kids."
)

## 4. Multilingual Support

The Navigator speaks English, Spanish, Hmong, and Somali. It detects the input language and responds in kind.

In [ ]:
# Spanish
print("=" * 90)
print("QUERY: Spanish - single mother needing food and rent help")
print("=" * 90)
response_es = ask_navigator(
    "Soy madre soltera con dos hijos. Perdí mi trabajo y necesito ayuda "
    "con comida y alquiler. Vivo en el condado de Dakota."
)

print("\n")

# Hmong
print("=" * 90)
print("QUERY: Hmong - elderly person needing heating assistance")
print("=" * 90)
response_hmn = ask_navigator(
    "Kuv yog ib tug neeg laus nyob hauv Hennepin County. "
    "Kuv xav tau kev pab them nqi cua sov rau lub caij ntuj no."
)

print("\n")

# Somali
print("=" * 90)
print("QUERY: Somali - family needing food assistance")
print("=" * 90)
response_so = ask_navigator(
    "Waxaan ahay qof qoys ah oo degan Ramsey County. "
    "Waxaan u baahanahay cunto caawimaad ah carruurta."
)

## 5. Readability Analysis

A core design goal is accessibility. We measure the reading level of model responses using Flesch-Kincaid grade level. Benefits information is typically written at a 12th-grade level or above; Navigator targets 5th-8th grade.

In [ ]:
import textstat

# Analyze readability of the English response
fk_grade = textstat.flesch_kincaid_grade(response_en)
flesch_ease = textstat.flesch_reading_ease(response_en)
smog = textstat.smog_index(response_en)
word_count = textstat.lexicon_count(response_en)
sentence_count = textstat.sentence_count(response_en)

print("Readability Analysis — English Response")
print("=" * 50)
print(f"  Flesch-Kincaid Grade Level:  {fk_grade:.1f}")
print(f"  Flesch Reading Ease:         {flesch_ease:.1f}")
print(f"  SMOG Index:                  {smog:.1f}")
print(f"  Word Count:                  {word_count}")
print(f"  Sentence Count:              {sentence_count}")
print()

if fk_grade <= 8:
    print(f"  Grade {fk_grade:.1f} — accessible to most adults (target: 5-8)")
elif fk_grade <= 12:
    print(f"  Grade {fk_grade:.1f} — high school level (acceptable)")
else:
    print(f"  Grade {fk_grade:.1f} — above target, may need simplification")

# Compare with typical government benefits text
gov_sample = (
    "Applicants must demonstrate financial eligibility as determined by the "
    "Federal Poverty Level guidelines established pursuant to Section 673(2) "
    "of the Community Services Block Grant Act. Categorical eligibility may "
    "be established through participation in Supplemental Nutrition Assistance "
    "Program benefits or Temporary Assistance for Needy Families."
)
gov_grade = textstat.flesch_kincaid_grade(gov_sample)
print(f"\n  Comparison — typical government text: Grade {gov_grade:.1f}")
print(f"  Navigator improvement: {gov_grade - fk_grade:.1f} grade levels more accessible")

## 6. Fine-Tuning Details

The base Gemma 4 E4B model was fine-tuned using **QLoRA** (4-bit quantized Low-Rank Adaptation) via **Unsloth** on 4x A100-80GB GPUs with DDP.

### Training Configuration

| Parameter | Value |
|---|---|
| Base model | `google/gemma-4-4b-it` |
| Method | QLoRA (4-bit NF4) |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Target modules | All linear layers |
| Training examples | ~10,000 (EN + ES + HMN + SO) |
| Epochs | 3 |
| Learning rate | 2e-4 (cosine decay) |
| Max sequence length | 2048 |
| Hardware | 4x A100-80GB (DDP) |
| Training time | ~45 minutes |
| Export | GGUF q4_k_m (5.3GB) |

### Training Data Sources

| Source | Documents | Content |
|---|---|---|
| DHS Combined Manual | 614 sections | Minnesota public assistance policies |
| County Programs | 267 programs | 5 counties + 3 CAP agencies |
| SAM.gov | 515 listings | Federal assistance programs |
| Generated (Gemini) | ~2,500 Q&A pairs | Multilingual training examples |

### Unsloth Optimization

Unsloth provided 2x training speedup over standard PEFT by:
- Automatic handling of Gemma 4's `ClippableLinear` layers (extends `nn.Module`, not `nn.Linear`)
- Fused LoRA kernels reducing memory overhead
- Native GGUF export without separate conversion step

## 7. Model Benchmark: Response Quality

Let's test the model on several diverse scenarios and evaluate response quality.

In [ ]:
test_scenarios = [
    {
        "name": "Elderly veteran - heating assistance",
        "query": "I'm a 68-year-old veteran in Hennepin County living on Social Security. "
                 "I'm having trouble paying my heating bill this winter.",
    },
    {
        "name": "Young adult - disability support",
        "query": "I'm 22 years old and was just diagnosed with a disability that prevents me "
                 "from working full time. I live in Dakota County with my parents but need to "
                 "find my own place. What help is available?",
    },
    {
        "name": "Immigrant family - food assistance",
        "query": "My family recently moved to Minnesota from another country. We have two "
                 "children in school. My husband works part-time making about $1,800 a month. "
                 "We live in Scott County. We need help with food and health insurance.",
    },
    {
        "name": "Domestic violence survivor",
        "query": "I need to leave an unsafe living situation with my 4-year-old. I don't have "
                 "a job or any money saved. I'm in Anoka County. What emergency help can I get?",
    },
]

results = []
for scenario in test_scenarios:
    print("=" * 90)
    print(f"SCENARIO: {scenario['name']}")
    print("=" * 90)
    resp = ask_navigator(scenario["query"])
    
    grade = textstat.flesch_kincaid_grade(resp)
    words = textstat.lexicon_count(resp)
    results.append({
        "scenario": scenario["name"],
        "grade_level": grade,
        "word_count": words,
        "has_disclaimer": "not legal advice" in resp.lower() or "informational" in resp.lower(),
        "says_may_eligible": "may be eligible" in resp.lower() or "may qualify" in resp.lower(),
    })
    print(f"\n  [Grade: {grade:.1f} | Words: {words} | Disclaimer: {results[-1]['has_disclaimer']}]")
    print()

In [ ]:
import pandas as pd

# Summary table
df = pd.DataFrame(results)
print("Response Quality Summary")
print("=" * 70)
print(df.to_string(index=False))
print()
print(f"Average reading grade level: {df['grade_level'].mean():.1f}")
print(f"Disclaimer present: {df['has_disclaimer'].sum()}/{len(df)}")
print(f"Uses 'may be eligible' phrasing: {df['says_may_eligible'].sum()}/{len(df)}")

## 8. Safety & Trust

NorthStar Navigator is designed with safety at its core:

- **Never says "qualifies"** — always uses "may be eligible" to avoid false promises
- **Disclaimer on every response** — reminds users this is informational, not legal advice
- **No personal data stored** — all inference runs locally via Ollama; no data leaves the device
- **Directs to 211** — when the model cannot help, it redirects to Minnesota's 211 helpline
- **Cites sources** — responses include program names, eligibility thresholds, and application portals
- **Reading level appropriate** — adjustable from 5th grade to 12th grade to match user literacy

### Why This Matters for Digital Equity

Minnesota has over 300,000 residents with limited English proficiency, concentrated among Hmong, Somali, and Spanish-speaking communities. Government benefits websites are typically:
- Written at a 12th+ grade reading level
- Available only in English (or with poor machine translation)
- Structured around program names, not human situations

NorthStar Navigator flips this: users describe their *situation*, and the system finds the *programs*. In their own language. At a reading level they can understand.

## 9. Ollama Integration

NorthStar Navigator uses Ollama as its inference backend, providing:

- **Local inference**: No API calls to external services. The model runs entirely on the user's GPU.
- **GGUF quantization**: The q4_k_m format reduces the 4B parameter model from ~8GB (fp16) to 5.3GB while maintaining quality.
- **Simple deployment**: `ollama create navigator -f Modelfile` + `ollama serve` = running model server.
- **Streaming support**: Ollama streams tokens as they're generated, enabling real-time chat UX.

### Ollama API Usage in Navigator

In [ ]:
# Show model info
model_info = ollama.show("navigator")
print("Model: navigator")
print(f"  Format:      {model_info.get('details', {}).get('format', 'N/A')}")
print(f"  Family:      {model_info.get('details', {}).get('family', 'N/A')}")
print(f"  Parameters:  {model_info.get('details', {}).get('parameter_size', 'N/A')}")
print(f"  Quantization:{model_info.get('details', {}).get('quantization_level', 'N/A')}")

# Demonstrate streaming
print("\nStreaming demo:")
print("-" * 50)
stream = ollama.chat(
    model="navigator",
    messages=[{"role": "user", "content": "What is SNAP in one sentence?"}],
    stream=True,
    options={"num_predict": 100},
)
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print()

## 10. Conclusion

NorthStar Navigator demonstrates that a fine-tuned Gemma 4 model, served locally through Ollama, can make government benefits information genuinely accessible to the people who need it most.

**What we built:**
- A three-stage NLP pipeline that converts plain-language descriptions of hardship into actionable program recommendations
- Support for four languages (English, Spanish, Hmong, Somali) covering Minnesota's largest LEP communities
- Responses written at a 5th-8th grade reading level, compared to the 12th+ grade level of typical government documents
- A RAG knowledge base of 1,938 documents covering Minnesota DHS policies, county programs, and federal assistance listings

**How we built it:**
- Fine-tuned Gemma 4 E4B with QLoRA via **Unsloth** on ~10K multilingual training examples
- Exported to GGUF q4_k_m (5.3GB) for efficient local inference via **Ollama**
- Deployed on RunPod GPU Pod with Gradio UI for the live demo

**Impact:**
- Bridges the language barrier for 300,000+ Minnesota residents with limited English proficiency
- Reduces the reading level gap between government documents and the people they serve
- Runs entirely locally — no personal data ever leaves the device
- Open source and reproducible

---

*NorthStar Navigator is informational only and does not constitute legal advice. Users should verify eligibility with the relevant county or state agency. For immediate assistance, dial 2-1-1.*